# project_tools

> MCP tools for project selection, discovery, and bookmarking

In [ ]:
#| default_exp tools.project

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional

import os
import json
import asyncio
import hashlib
from pathlib import Path

from mcp.server.fastmcp import FastMCP
from mcp.types import ToolAnnotations

from nbdev_mcp.utils.config import load_bookmarks, save_bookmarks
from nbdev_mcp.utils.paths import (
    expand,
    project_summary,
    is_nbdev_project,
    resolve_selector,
    require_project,
    settings_dict,
    nbdev_settings_path,
    nbdev_generation,
)
from nbdev_mcp.utils.rich import render_result, render_table


## Project Selection

In [ ]:
#| export
def set_project(selector: str) -> Dict[str, Any]:
    """Tool: Select an nbdev project to make it active (by path or alias).
    
    Args:
        selector: Project path, alias (prefixed with @), or bookmark name.
    
    Returns:
        Result dict with 'ok', 'project' info, and 'pretty' formatted output.
    """
    try:
        p = resolve_selector(selector)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    # Use set_current_project to update both config and paths modules
    from nbdev_mcp.utils.config import set_current_project
    set_current_project(p)
    
    meta = project_summary(p)
    pretty = render_result('Project selected', meta)
    return {'ok': True, 'project': meta, 'pretty': pretty}

In [ ]:
#| export
def current_project() -> Dict[str, Any]:
    """Tool: Show the currently active project's summary information.
    
    Returns:
        Result dict with 'ok', 'project' info, and 'pretty' formatted output.
    
    Raises:
        RuntimeError: If no project is currently selected.
    """
    p = require_project()
    meta = project_summary(p)
    pretty = render_result('Current project', meta)
    return {'ok': True, 'project': meta, 'pretty': pretty}

In [ ]:
#| export
def console_scripts_status(project: Optional[str] = None) -> Dict[str, Any]:
    """Tool: Show console_scripts entry points and where to configure them.
    
    Args:
        project: Project path or alias. Uses current project if not specified.
    
    Returns:
        Result dict with 'entries' list and suggestions for adding scripts.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}

    settings = settings_dict(p)
    settings_file = nbdev_settings_path(p)
    settings_name = settings_file.name if settings_file is not None else 'settings.ini'

    cs = settings.get('console_scripts', '').strip()
    entries = [entry for entry in cs.split() if entry] if cs else []

    if entries:
        msg = f'console_scripts present in {settings_name}.'
    else:
        if settings_name.endswith('.toml'):
            example = 'console_scripts = ["mycli=mypkg:main"]'
        else:
            example = 'console_scripts = mycli=mypkg:main'
        msg = f'No console_scripts configured. Add e.g. `{example}` in {settings_name}.'

    pretty = render_result('console_scripts', {'entries': entries or 'None', 'settings_file': settings_name}, {})
    return {
        'ok': True,
        'entries': entries,
        'settings_file': settings_name,
        'nbdev_generation': nbdev_generation(p),
        'message': msg,
        'pretty': pretty,
    }


## Project Discovery

In [ ]:
#| export
def find_projects(
    roots: Optional[List[str]] = None,
    max_results: int = 50
) -> Dict[str, Any]:
    """Tool: Scan given directories (or common defaults) for nbdev projects.
    
    Searches in provided roots, NBDEV_PROJECTS env var, and common
    directories like ~/code, ~/projects, ~/repos, ~/src, ~/Dev.
    
    Args:
        roots: Additional directories to scan.
        max_results: Maximum number of projects to return (default 50).
    
    Returns:
        Result dict with 'results' list of project summaries.
    """
    search_dirs: List[Path] = []
    
    if roots:
        search_dirs += [expand(r) for r in roots]
    
    env = os.environ.get('NBDEV_PROJECTS')
    if env:
        for r in env.split(os.pathsep):
            pr = expand(r)
            if pr.exists():
                search_dirs.append(pr)
    
    home = Path.home()
    for sub in ('code', 'projects', 'repos', 'src', 'Dev', 'dev', 'Documents'):
        d = home / sub
        if d.exists():
            search_dirs.append(d)
    
    seen, results = set(), []
    for base in search_dirs:
        if not base.is_dir():
            continue
        for p in base.iterdir():
            if p.is_dir() and p not in seen:
                try:
                    if is_nbdev_project(p):
                        results.append(project_summary(p))
                        seen.add(p)
                except Exception:
                    continue
        if len(results) >= max_results:
            break
    
    rows = [[r.get('lib_name') or '?', r['project'], r['nbs_dir']] for r in results]
    pretty = render_table('Discovered nbdev projects', ['lib_name', 'path', 'nbs_dir'], rows)
    
    return {'ok': True, 'results': results, 'pretty': pretty}

## Bookmarks

In [ ]:
#| export
def bookmark_project(alias: str, path: str) -> Dict[str, Any]:
    """Tool: Bookmark an nbdev project path with a short alias name.
    
    Args:
        alias: Short name to use as bookmark (e.g., 'myproj').
        path: Path to the nbdev project directory.
    
    Returns:
        Result dict with 'alias' and 'path' of the saved bookmark.
    """
    try:
        root = resolve_selector(path)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    aliases = load_bookmarks()
    aliases[alias] = str(root)
    save_bookmarks(aliases)
    
    meta = {'alias': alias, 'path': str(root)}
    pretty = render_result('Bookmarked project', meta)
    return {'ok': True, **meta, 'pretty': pretty}

In [ ]:
#| export
def list_bookmarks() -> Dict[str, Any]:
    """Tool: List all saved project bookmarks (alias -> path).
    
    Returns:
        Result dict with 'aliases' mapping bookmark names to project paths.
    """
    aliases = load_bookmarks()
    rows = [[k, v] for k, v in aliases.items()]
    pretty = render_table('Project bookmarks', ['alias', 'path'], rows)
    return {'ok': True, 'aliases': aliases, 'pretty': pretty}

In [ ]:
#| export
def remove_bookmark(alias: str) -> Dict[str, Any]:
    """Tool: Remove a saved project bookmark by alias.
    
    Args:
        alias: Bookmark name to remove.
    
    Returns:
        Result dict with removed alias and path, or error if not found.
    """
    aliases = load_bookmarks()
    if alias in aliases:
        path = aliases.pop(alias)
        save_bookmarks(aliases)
        meta = {'alias': alias, 'removed': path}
        pretty = render_result('Removed bookmark', meta)
        return {'ok': True, **meta, 'pretty': pretty}
    return {'ok': False, 'error': f'No such alias: {alias}'}

In [ ]:
#| export
def config_status() -> Dict[str, Any]:
    """Tool: Show current nbdev-mcp configuration settings.

    Returns:
        Result dict with 'config' settings, 'bookmarks_path', and 'env_overrides'.
    """
    from nbdev_mcp.utils.config import get_config, BOOKMARKS_PATH

    cfg = get_config()
    config_data = {
        'log_level': cfg.get('log_level', 'INFO'),
        'prompt_dir': cfg.get('prompt_dir', 'prompt_templates'),
        'env_dir_name': cfg.get('env_dir_name', 'Environments'),
        'max_tree_files': cfg.get('max_tree_files', 600),
    }

    # Check for environment variable overrides
    env_overrides = {}
    for key in ['LOG_LEVEL', 'PROMPT_DIR', 'ENV_DIR_NAME', 'MAX_TREE_FILES']:
        env_key = f'NBMCP_{key}'
        if env_key in os.environ:
            env_overrides[env_key] = os.environ[env_key]

    pretty = render_result('Configuration', config_data, {
        'bookmarks_path': str(BOOKMARKS_PATH),
        'env_overrides': env_overrides or 'None'
    })

    return {
        'ok': True,
        'config': config_data,
        'bookmarks_path': str(BOOKMARKS_PATH),
        'env_overrides': env_overrides,
        'pretty': pretty
    }

In [ ]:
#| export
def prompt_templates_status() -> Dict[str, Any]:
    """Tool: List available prompt templates and their locations.

    Returns:
        Result dict with 'templates' list and 'prompt_dir' setting.
    """
    from nbdev_mcp.utils.config import get_config, EXPECTED_PROMPT_TEMPLATES
    from nbdev_mcp.prompts import get_bundled_template

    cfg = get_config()
    prompt_dir = cfg.get('prompt_dir', 'prompt_templates')

    # List of expected templates
    template_names = EXPECTED_PROMPT_TEMPLATES

    templates = []
    for name in template_names:
        try:
            content = get_bundled_template(name)
            templates.append({
                'name': name,
                'path': f'nbdev_mcp.prompt_templates/{name}',
                'exists': True,
                'size': len(content)
            })
        except FileNotFoundError:
            templates.append({
                'name': name,
                'path': 'not found',
                'exists': False,
                'size': 0
            })

    rows = [[t['name'], t['path'], '✓' if t['exists'] else '✗'] for t in templates]
    pretty = render_table('Prompt Templates', ['name', 'path', 'exists'], rows)

    return {
        'ok': True,
        'templates': templates,
        'prompt_dir': prompt_dir,
        'pretty': pretty
    }

In [ ]:
#| export
def health_check() -> Dict[str, Any]:
    """Tool: Validate MCP connection and report system status.
    
    Returns status of:
    - MCP server connection
    - Current project (if any)
    - Key dependencies
    - Available tools count
    
    Returns
    -------
    Dict[str, Any]
        Result with health status, version info, and diagnostics.
    """
    import sys
    import platform
    
    # Check package versions
    deps = {}
    for pkg in ['nbdev', 'mcp', 'fastcore', 'rich']:
        try:
            from importlib.metadata import version as pkg_version
            deps[pkg] = pkg_version(pkg)
        except Exception:
            deps[pkg] = 'not installed'
    
    # Get nbdev-mcp version
    try:
        from importlib.metadata import version as pkg_version
        nbdev_mcp_version = pkg_version('nbdev-mcp')
    except Exception:
        nbdev_mcp_version = 'unknown'
    
    # Check current project
    from nbdev_mcp.utils.config import CURRENT_PROJECT
    project_status = {
        'has_project': CURRENT_PROJECT is not None,
        'path': str(CURRENT_PROJECT) if CURRENT_PROJECT else None
    }
    
    # Build health report
    health = {
        'status': 'healthy',
        'version': nbdev_mcp_version,
        'python': sys.version.split()[0],
        'platform': platform.system(),
        'dependencies': deps,
        'project': project_status,
    }
    
    # Check for issues
    issues = []
    if deps.get('nbdev') == 'not installed':
        issues.append('nbdev is not installed')
        health['status'] = 'degraded'
    if deps.get('mcp') == 'not installed':
        issues.append('mcp is not installed')
        health['status'] = 'error'
    
    if issues:
        health['issues'] = issues
    
    from nbdev_mcp.utils.rich import render_result
    pretty = render_result('Health Check', health)
    return {'ok': health['status'] != 'error', **health, 'pretty': pretty}

In [ ]:
#| export
def split_source_lines(source: str) -> List[str]:
    """Split notebook cell source into newline-preserving lines."""
    if source == '':
        return []
    return source.splitlines(keepends=True)


def make_markdown_cell(source: str) -> Dict[str, Any]:
    """Create a markdown notebook cell payload."""
    return {
        'cell_type': 'markdown',
        'metadata': {},
        'source': split_source_lines(source),
    }


def make_code_cell(source: str) -> Dict[str, Any]:
    """Create a code notebook cell payload."""
    return {
        'cell_type': 'code',
        'metadata': {},
        'execution_count': None,
        'outputs': [],
        'source': split_source_lines(source),
    }


def make_notebook_payload(cells: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Create a minimal notebook payload."""
    return {
        'cells': cells,
        'metadata': {
            'kernelspec': {
                'display_name': 'Python 3',
                'language': 'python',
                'name': 'python3',
            },
            'language_info': {'name': 'python'},
        },
        'nbformat': 4,
        'nbformat_minor': 5,
    }


def build_mcp_scaffold_plan(
    server_name: str = 'mcp.custom',
    module_prefix: str = '50_mcp',
    package_name: str = 'mcp_custom',
) -> Dict[str, Any]:
    """Build a notebook-first MCP scaffold plan."""
    notebooks = [
        {
            'path': f'nbs/{module_prefix}/00__init__.ipynb',
            'default_exp': f'{package_name}.mcp_init',
            'purpose': 'Module bootstrap and exports',
        },
        {
            'path': f'nbs/{module_prefix}/01_server.ipynb',
            'default_exp': f'{package_name}.server',
            'purpose': 'FastMCP server composition root',
        },
        {
            'path': f'nbs/{module_prefix}/02_tools.ipynb',
            'default_exp': f'{package_name}.tools',
            'purpose': 'Tool registration and implementations',
        },
        {
            'path': f'nbs/{module_prefix}/03_prompts.ipynb',
            'default_exp': f'{package_name}.prompts',
            'purpose': 'Prompt definitions and prompt registry',
        },
        {
            'path': f'nbs/{module_prefix}/04_cli.ipynb',
            'default_exp': f'{package_name}.cli',
            'purpose': 'CLI entrypoint and run helpers',
        },
    ]

    settings_patch = {
        'lib_name': package_name,
        'nbs_path': 'nbs',
        'console_scripts': f'{package_name}={package_name}.cli:main',
    }

    checklist = [
        'Create notebooks under nbs/ with one responsibility per notebook.',
        'Set #| default_exp and #| export cells for public API.',
        'Compose server in one create_server(...) function.',
        'Wire add_*_tools and add_prompts from exported modules.',
        'Expose CLI via settings console_scripts.',
        'Run nbdev_export and test before install.',
    ]

    return {
        'server_name': server_name,
        'module_prefix': module_prefix,
        'package_name': package_name,
        'notebooks': notebooks,
        'settings_patch': settings_patch,
        'checklist': checklist,
    }


def mcp_scaffold_guide(
    server_name: str = 'mcp.custom',
    module_prefix: str = '50_mcp',
    package_name: str = 'mcp_custom',
) -> Dict[str, Any]:
    """Tool: Return a notebook-first scaffold guide for building a new MCP server.

    The guide follows this project's conventions:
    - All scripting logic lives under ``nbs/``
    - Exported APIs are surfaced via nbdev settings
    - Generated ``.py`` modules are build artifacts
    """
    plan = build_mcp_scaffold_plan(server_name=server_name, module_prefix=module_prefix, package_name=package_name)
    pretty = render_result('MCP Scaffold Guide', plan)
    return {'ok': True, **plan, 'pretty': pretty}


def scaffold_mcp_notebooks(
    project: Optional[str] = None,
    server_name: str = 'mcp.custom',
    module_prefix: str = '50_mcp',
    package_name: str = 'mcp_custom',
    dry_run: bool = True,
    overwrite: bool = False,
) -> Dict[str, Any]:
    """Tool: Create notebook scaffold files for a new MCP module.

    Parameters
    ----------
    project : str, optional
        Project path or alias.
    server_name : str
        FastMCP server name.
    module_prefix : str
        Notebook folder under nbs/ (e.g. ``50_mcp``).
    package_name : str
        Package/module base name for generated ``default_exp`` values.
    dry_run : bool
        If True, report what would be created without writing files.
    overwrite : bool
        If True, overwrite existing notebooks.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}

    plan = build_mcp_scaffold_plan(server_name=server_name, module_prefix=module_prefix, package_name=package_name)
    target_dir = p / 'nbs' / module_prefix

    notebook_templates: Dict[str, Dict[str, Any]] = {}
    for item in plan['notebooks']:
        nb_path = p / item['path']
        default_exp = item['default_exp']
        purpose = item['purpose']

        if item['path'].endswith('00__init__.ipynb'):
            code_body = (
                f"#| export\n"
                f"from mcp.server.fastmcp import FastMCP\n\n"
                f"def create_server(name: str = '{server_name}') -> FastMCP:\n"
                f"    \"\"\"Create base MCP server.\"\"\"\n"
                f"    return FastMCP(name)\n"
            )
        elif item['path'].endswith('01_server.ipynb'):
            code_body = (
                f"#| export\n"
                f"from {package_name}.mcp_init import create_server\n"
                f"from {package_name}.tools import add_tools\n"
                f"from {package_name}.prompts import add_prompts\n\n"
                f"def build_server(name: str = '{server_name}'):\n"
                f"    server = create_server(name)\n"
                f"    add_tools(server)\n"
                f"    add_prompts(server)\n"
                f"    return server\n"
            )
        elif item['path'].endswith('02_tools.ipynb'):
            code_body = (
                "#| export\n"
                "def add_tools(mcp):\n"
                "    @mcp.tool()\n"
                "    def health() -> dict:\n"
                "        return {'ok': True, 'message': 'healthy'}\n"
            )
        elif item['path'].endswith('03_prompts.ipynb'):
            code_body = (
                "#| export\n"
                "def add_prompts(mcp):\n"
                "    @mcp.prompt()\n"
                "    def system_prompt() -> str:\n"
                "        return 'You are an MCP agent.'\n"
            )
        else:
            code_body = (
                f"#| export\n"
                f"from {package_name}.server import build_server\n\n"
                f"def main() -> None:\n"
                f"    build_server().run(transport='stdio')\n"
            )

        notebook_templates[str(nb_path)] = make_notebook_payload([
            make_markdown_cell(f"# {purpose}\n"),
            make_code_cell(f"#| default_exp {default_exp}\n"),
            make_code_cell(code_body),
        ])

    existing = [path for path in notebook_templates.keys() if Path(path).exists()]
    to_create = [path for path in notebook_templates.keys() if path not in existing or overwrite]
    blocked = [path for path in existing if not overwrite]

    if dry_run:
        result = {
            'ok': True,
            'project': str(p),
            'target_dir': str(target_dir),
            'server_name': server_name,
            'module_prefix': module_prefix,
            'package_name': package_name,
            'create_count': len(to_create),
            'create_paths': to_create,
            'blocked_existing': blocked,
        }
        pretty = render_result('MCP Notebook Scaffold (dry-run)', result)
        return {**result, 'pretty': pretty}

    target_dir.mkdir(parents=True, exist_ok=True)
    created: List[str] = []
    for nb_path, payload in notebook_templates.items():
        path_obj = Path(nb_path)
        if path_obj.exists() and not overwrite:
            continue
        path_obj.parent.mkdir(parents=True, exist_ok=True)
        path_obj.write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')
        created.append(nb_path)

    result = {
        'ok': True,
        'project': str(p),
        'created_count': len(created),
        'created_paths': created,
        'blocked_existing': blocked,
        'next_steps': [
            'Run nbdev_export',
            'Add module imports in nbs/11_tools/00__init__.ipynb if needed',
            'Add tests for new tools/resources/prompts',
        ],
    }
    pretty = render_result('MCP Notebook Scaffold', result)
    return {**result, 'pretty': pretty}


def normalize_json_value(value: Any) -> Any:
    """Normalize values for deterministic JSON serialization."""
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, list):
        return [normalize_json_value(v) for v in value]
    if isinstance(value, dict):
        return {str(k): normalize_json_value(v) for k, v in value.items()}
    if hasattr(value, 'model_dump'):
        return normalize_json_value(value.model_dump())
    if hasattr(value, 'dict'):
        return normalize_json_value(value.dict())
    return str(value)


def schema_hash(value: Any) -> str:
    """Create a stable hash for schema-like payloads."""
    normalized = normalize_json_value(value)
    encoded = json.dumps(normalized, sort_keys=True, separators=(',', ':'))
    return hashlib.sha256(encoded.encode('utf-8')).hexdigest()


async def collect_contract_async(server_name: str = 'mcp.nbdev') -> Dict[str, Any]:
    """Collect MCP server contract data asynchronously."""
    from nbdev_mcp.mcp import create_nbdev_mcp

    server = create_nbdev_mcp(name=server_name)
    tools = await server.list_tools()
    resources = await server.list_resources()
    prompts = await server.list_prompts()

    tool_items = []
    for tool in tools:
        input_schema = normalize_json_value(getattr(tool, 'inputSchema', {}))
        output_schema = normalize_json_value(getattr(tool, 'outputSchema', {}))
        tool_items.append({
            'name': tool.name,
            'description': tool.description,
            'input_schema': input_schema,
            'output_schema': output_schema,
            'input_schema_hash': schema_hash(input_schema),
            'output_schema_hash': schema_hash(output_schema),
        })

    resource_items = []
    for resource in resources:
        resource_items.append({
            'name': getattr(resource, 'name', ''),
            'uri': str(getattr(resource, 'uri', '')),
            'description': getattr(resource, 'description', ''),
        })

    prompt_items = []
    for prompt in prompts:
        prompt_items.append({
            'name': getattr(prompt, 'name', ''),
            'description': getattr(prompt, 'description', ''),
            'arguments': normalize_json_value(getattr(prompt, 'arguments', [])),
        })

    signature = {
        'tools': [{'name': t['name'], 'input_schema_hash': t['input_schema_hash'], 'output_schema_hash': t['output_schema_hash']} for t in sorted(tool_items, key=lambda x: x['name'])],
        'resources': sorted([r['uri'] for r in resource_items]),
        'prompts': sorted([p['name'] for p in prompt_items]),
    }

    contract_hash = schema_hash(signature)

    from datetime import datetime, timezone
    return {
        'server_name': server_name,
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'tool_count': len(tool_items),
        'resource_count': len(resource_items),
        'prompt_count': len(prompt_items),
        'tools': sorted(tool_items, key=lambda x: x['name']),
        'resources': sorted(resource_items, key=lambda x: x['uri']),
        'prompts': sorted(prompt_items, key=lambda x: x['name']),
        'contract_hash': contract_hash,
    }


def mcp_contract_snapshot(
    server_name: str = 'mcp.nbdev',
    output_path: str = 'contracts/mcp_contract.json',
    project: Optional[str] = None,
    write_file: bool = False,
) -> Dict[str, Any]:
    """Tool: Generate an MCP contract snapshot for compatibility checks."""
    try:
        contract = asyncio.run(collect_contract_async(server_name=server_name))
    except RuntimeError as e:
        return {'ok': False, 'error': f'Could not collect contract in current event loop: {e}'}
    except Exception as e:
        return {'ok': False, 'error': str(e)}

    result = {
        'ok': True,
        'contract': contract,
        'server_name': server_name,
        'contract_hash': contract['contract_hash'],
    }

    if write_file:
        base = resolve_selector(project) if project else Path.cwd()
        target = (base / output_path).resolve()
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(json.dumps(result, indent=2) + '\n', encoding='utf-8')
        result['output_path'] = str(target)

    pretty_meta = {
        'server_name': server_name,
        'tool_count': contract['tool_count'],
        'resource_count': contract['resource_count'],
        'prompt_count': contract['prompt_count'],
        'contract_hash': contract['contract_hash'][:12],
        'write_file': write_file,
        'output_path': result.get('output_path', 'not written'),
    }
    result['pretty'] = render_result('MCP Contract Snapshot', pretty_meta)
    return result


def mcp_contract_diff(
    baseline_path: str,
    current_path: Optional[str] = None,
    server_name: str = 'mcp.nbdev',
) -> Dict[str, Any]:
    """Tool: Compare a baseline MCP contract against current contract."""
    baseline_file = expand(baseline_path)
    if not baseline_file.exists():
        return {'ok': False, 'error': f'Baseline not found: {baseline_file}'}

    baseline_payload = json.loads(baseline_file.read_text(encoding='utf-8'))
    baseline_contract = baseline_payload.get('contract', baseline_payload)

    if current_path:
        current_file = expand(current_path)
        if not current_file.exists():
            return {'ok': False, 'error': f'Current contract not found: {current_file}'}
        current_payload = json.loads(current_file.read_text(encoding='utf-8'))
        current_contract = current_payload.get('contract', current_payload)
    else:
        current_snapshot = mcp_contract_snapshot(server_name=server_name, write_file=False)
        if not current_snapshot.get('ok'):
            return current_snapshot
        current_contract = current_snapshot['contract']

    baseline_tools = {t['name']: t for t in baseline_contract.get('tools', [])}
    current_tools = {t['name']: t for t in current_contract.get('tools', [])}

    baseline_tool_names = set(baseline_tools)
    current_tool_names = set(current_tools)

    added_tools = sorted(current_tool_names - baseline_tool_names)
    removed_tools = sorted(baseline_tool_names - current_tool_names)

    changed_tool_schemas: List[str] = []
    for name in sorted(baseline_tool_names & current_tool_names):
        base_schema = baseline_tools[name].get('input_schema_hash')
        curr_schema = current_tools[name].get('input_schema_hash')
        base_output = baseline_tools[name].get('output_schema_hash')
        curr_output = current_tools[name].get('output_schema_hash')
        if base_schema != curr_schema or base_output != curr_output:
            changed_tool_schemas.append(name)

    baseline_resources = {r.get('uri', '') for r in baseline_contract.get('resources', [])}
    current_resources = {r.get('uri', '') for r in current_contract.get('resources', [])}
    added_resources = sorted(current_resources - baseline_resources)
    removed_resources = sorted(baseline_resources - current_resources)

    baseline_prompts = {p.get('name', '') for p in baseline_contract.get('prompts', [])}
    current_prompts = {p.get('name', '') for p in current_contract.get('prompts', [])}
    added_prompts = sorted(current_prompts - baseline_prompts)
    removed_prompts = sorted(baseline_prompts - current_prompts)

    breaking = bool(removed_tools or changed_tool_schemas or removed_resources or removed_prompts)

    result = {
        'ok': True,
        'breaking': breaking,
        'added_tools': added_tools,
        'removed_tools': removed_tools,
        'changed_tool_schemas': changed_tool_schemas,
        'added_resources': added_resources,
        'removed_resources': removed_resources,
        'added_prompts': added_prompts,
        'removed_prompts': removed_prompts,
        'baseline_hash': baseline_contract.get('contract_hash', ''),
        'current_hash': current_contract.get('contract_hash', ''),
    }
    result['pretty'] = render_result('MCP Contract Diff', result)
    return result


def provider_entry_matches_expected(provider_name: str, expected: Dict[str, Any], actual: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """Compare required provider entry keys for drift checks."""
    if not isinstance(actual, dict):
        return {'matches': False, 'mismatches': ['missing entry']}

    keys = ['command', 'args']
    if provider_name in {'vscode', 'cursor'}:
        keys = ['type', 'command', 'args']

    mismatches: List[str] = []
    for key in keys:
        if actual.get(key) != expected.get(key):
            mismatches.append(f"{key}: expected {expected.get(key)!r}, got {actual.get(key)!r}")

    return {'matches': len(mismatches) == 0, 'mismatches': mismatches}


def mcp_provider_drift_report(provider: Optional[str] = None) -> Dict[str, Any]:
    """Tool: Report drift between provider config and expected nbdev-mcp config."""
    import nbdev_mcp.mcp as mcp_module

    if provider:
        try:
            providers = [mcp_module.Provider(provider.lower())]
        except Exception:
            valid = ', '.join(p.value for p in mcp_module.Provider)
            return {'ok': False, 'error': f'Unknown provider: {provider}. Valid: {valid}'}
    else:
        providers = list(mcp_module.Provider)

    reports: List[Dict[str, Any]] = []

    for provider_enum in providers:
        status = mcp_module.check_provider_status(provider_enum)
        expected = mcp_module.make_server_config_for_provider(provider_enum, auto_start=False)
        suggestion = f"python -m nbdev_mcp update {provider_enum.value} --strategy merge --dry-run"

        report = {
            'provider': provider_enum.value,
            'exists': status.get('exists', False),
            'installed': status.get('installed', False),
            'path': status.get('path', ''),
            'format': status.get('format', 'json'),
            'drifted': False,
            'mismatches': [],
            'suggestion': suggestion,
        }

        if not report['exists']:
            report['drifted'] = True
            report['mismatches'] = ['config missing']
            reports.append(report)
            continue

        try:
            config_path = Path(report['path'])
            config_format = report['format']
            config_text = config_path.read_text(encoding='utf-8')

            if config_format == 'toml':
                config = mcp_module.parse_toml(config_text)
                actual_entry = config.get('mcp_servers', {}).get('nbdev')
            else:
                config = mcp_module.parse_jsonc(config_text)
                mcp_key = mcp_module.get_mcp_key(provider_enum, config_format=config_format)
                actual_entry = config.get(mcp_key, {}).get('nbdev') if isinstance(config.get(mcp_key), dict) else None

            compare = provider_entry_matches_expected(provider_enum.value, expected, actual_entry)
            report['drifted'] = not compare['matches']
            report['mismatches'] = compare['mismatches']
        except Exception as e:
            report['drifted'] = True
            report['mismatches'] = [f'parse/read error: {e}']

        reports.append(report)

    rows = [[r['provider'], 'yes' if r['drifted'] else 'no', '; '.join(r['mismatches']) or '-', r['path']] for r in reports]
    pretty = render_table('Provider Drift Report', ['provider', 'drifted', 'mismatches', 'path'], rows)

    return {
        'ok': True,
        'providers': reports,
        'drifted_count': sum(1 for r in reports if r['drifted']),
        'pretty': pretty,
    }


def mcp_composition_workbench(
    local_servers: int = 1,
    remote_servers: int = 0,
    requires_transforms: bool = True,
    requires_strict_auth: bool = False,
    latency_budget_ms: int = 250,
) -> Dict[str, Any]:
    """Tool: Recommend MCP composition strategy beyond basic FastMCP setup."""
    recommendations: List[str] = []
    architecture: List[str] = []

    if local_servers > 1:
        architecture.append('Use mount-based composition for local servers.')
        recommendations.append('Apply namespace transforms to avoid tool collisions.')

    if remote_servers > 0:
        architecture.append('Use proxy/provider composition for remote servers.')
        recommendations.append('Add timeout/retry policy and health checks for remote links.')

    if requires_transforms:
        recommendations.append('Model transform chain explicitly: visibility -> namespace -> schema transforms.')

    if requires_strict_auth:
        recommendations.append('Use authorization layers and component visibility gates per server boundary.')

    if latency_budget_ms <= 150:
        recommendations.append('Favor local mounts and minimize proxy hops to keep latency within strict budget.')
    elif latency_budget_ms >= 500:
        recommendations.append('Latency budget allows richer proxy/middleware layers and contract checks in runtime path.')

    if not architecture:
        architecture.append('Single-server FastMCP setup is sufficient; add NBDevMCP only for CI/contracts/ops features.')

    plan = {
        'local_servers': local_servers,
        'remote_servers': remote_servers,
        'requires_transforms': requires_transforms,
        'requires_strict_auth': requires_strict_auth,
        'latency_budget_ms': latency_budget_ms,
        'architecture': architecture,
        'recommendations': recommendations,
        'next_actions': [
            'Generate scaffold with mcp_scaffold_guide / scaffold_mcp_notebooks.',
            'Capture baseline contract with mcp_contract_snapshot(write_file=True).',
            'Track drift with mcp_provider_drift_report and update dry-runs.',
        ],
    }
    pretty = render_result('MCP Composition Workbench', plan)
    return {'ok': True, **plan, 'pretty': pretty}


## MCP Governance and Compatibility


In [ ]:
#| export
def mcp_compatibility_matrix(
    project: Optional[str] = None,
    provider: Optional[str] = None,
) -> Dict[str, Any]:
    """Tool: Summarize provider/client compatibility and readiness."""
    if provider is not None:
        provider = provider.strip().lower()

    drift_report = mcp_provider_drift_report(provider=provider)
    if not drift_report.get('ok'):
        return drift_report

    provider_profiles: Dict[str, Dict[str, Any]] = {
        'codex': {
            'client': 'Codex',
            'preferred_config': 'toml',
            'auto_start_supported': False,
            'notes': 'Prefer TOML config; legacy JSON fallback is supported.',
        },
        'claude': {
            'client': 'Claude Code/Desktop',
            'preferred_config': 'json',
            'auto_start_supported': False,
            'notes': 'JSON config with a stdio command is the stable path.',
        },
        'vscode': {
            'client': 'VS Code',
            'preferred_config': 'json',
            'auto_start_supported': True,
            'notes': 'Supports chat.mcp.autostart in settings.json.',
        },
        'cursor': {
            'client': 'Cursor',
            'preferred_config': 'json',
            'auto_start_supported': True,
            'notes': 'Supports chat.mcp.autostart in settings.json.',
        },
    }

    generation = 'unknown'
    if project is not None:
        try:
            generation = nbdev_generation(resolve_selector(project))
        except Exception as e:
            return {'ok': False, 'error': str(e)}
    else:
        try:
            generation = nbdev_generation(resolve_selector(None))
        except Exception:
            generation = 'unknown'

    matrix: List[Dict[str, Any]] = []
    recommendations: List[str] = []

    for report in drift_report.get('providers', []):
        provider_name = report.get('provider', '')
        profile_data = provider_profiles.get(provider_name, {
            'client': provider_name,
            'preferred_config': report.get('format', 'json'),
            'auto_start_supported': False,
            'notes': '',
        })

        if not report.get('exists', False):
            readiness = 'missing_config'
            action = f"python -m nbdev_mcp install {provider_name}"
        elif not report.get('installed', False):
            readiness = 'not_installed'
            action = f"python -m nbdev_mcp install {provider_name}"
        elif report.get('drifted', False):
            readiness = 'drifted'
            action = report.get('suggestion', f"python -m nbdev_mcp update {provider_name} --strategy merge --dry-run")
        else:
            readiness = 'ready'
            action = 'No action required.'

        row = {
            'provider': provider_name,
            'client': profile_data['client'],
            'readiness': readiness,
            'installed': report.get('installed', False),
            'drifted': report.get('drifted', False),
            'config_format': report.get('format', 'json'),
            'preferred_config': profile_data['preferred_config'],
            'auto_start_supported': profile_data['auto_start_supported'],
            'notes': profile_data['notes'],
            'recommended_action': action,
            'path': report.get('path', ''),
        }
        matrix.append(row)

        if readiness in {'missing_config', 'not_installed'}:
            recommendations.append(f"Install provider config for {provider_name}: {action}")
        elif readiness == 'drifted':
            recommendations.append(f"Reconcile drift for {provider_name}: {action}")

    if generation == 'v2':
        recommendations.append('Project appears to use nbdev v2: use nbdev_prepare / nbdev_export command names.')
    elif generation == 'v3':
        recommendations.append('Project appears to use nbdev v3: use nbdev-prepare / nbdev-export command names.')

    if not recommendations:
        recommendations.append('Provider configs are aligned. Capture a new contract snapshot for CI baseline.')

    ready_count = sum(1 for row in matrix if row['readiness'] == 'ready')
    rows = [[
        row['provider'],
        row['readiness'],
        'yes' if row['installed'] else 'no',
        'yes' if row['drifted'] else 'no',
        row['config_format'],
        row['recommended_action'],
    ] for row in matrix]
    pretty = render_table(
        'MCP Compatibility Matrix',
        ['provider', 'readiness', 'installed', 'drifted', 'config', 'recommended_action'],
        rows,
    )

    return {
        'ok': True,
        'provider': provider,
        'project': project,
        'nbdev_generation': generation,
        'ready_count': ready_count,
        'total': len(matrix),
        'matrix': matrix,
        'recommendations': recommendations,
        'pretty': pretty,
    }


def mcp_contract_ci_gate(
    baseline_path: str,
    current_path: Optional[str] = None,
    server_name: str = 'mcp.nbdev',
    allow_additive_tools: bool = True,
    allow_additive_resources: bool = True,
    allow_additive_prompts: bool = True,
) -> Dict[str, Any]:
    """Tool: Enforce contract-compatibility gate semantics for CI."""
    diff = mcp_contract_diff(
        baseline_path=baseline_path,
        current_path=current_path,
        server_name=server_name,
    )
    if not diff.get('ok'):
        return diff

    violations: List[str] = []
    notices: List[str] = []

    removed_tools = diff.get('removed_tools', [])
    changed_tool_schemas = diff.get('changed_tool_schemas', [])
    removed_resources = diff.get('removed_resources', [])
    removed_prompts = diff.get('removed_prompts', [])

    added_tools = diff.get('added_tools', [])
    added_resources = diff.get('added_resources', [])
    added_prompts = diff.get('added_prompts', [])

    if removed_tools:
        violations.append(f"Removed tools are breaking: {', '.join(removed_tools)}")
    if changed_tool_schemas:
        violations.append(f"Tool schema changes are breaking: {', '.join(changed_tool_schemas)}")
    if removed_resources:
        violations.append(f"Removed resources are breaking: {', '.join(removed_resources)}")
    if removed_prompts:
        violations.append(f"Removed prompts are breaking: {', '.join(removed_prompts)}")

    if added_tools and not allow_additive_tools:
        violations.append(f"Added tools are disallowed in this gate: {', '.join(added_tools)}")
    elif added_tools:
        notices.append(f"Additive tools allowed: {', '.join(added_tools)}")

    if added_resources and not allow_additive_resources:
        violations.append(f"Added resources are disallowed in this gate: {', '.join(added_resources)}")
    elif added_resources:
        notices.append(f"Additive resources allowed: {', '.join(added_resources)}")

    if added_prompts and not allow_additive_prompts:
        violations.append(f"Added prompts are disallowed in this gate: {', '.join(added_prompts)}")
    elif added_prompts:
        notices.append(f"Additive prompts allowed: {', '.join(added_prompts)}")

    passed = len(violations) == 0
    summary = {
        'breaking': diff.get('breaking', False),
        'added_tools': len(added_tools),
        'removed_tools': len(removed_tools),
        'changed_tool_schemas': len(changed_tool_schemas),
        'added_resources': len(added_resources),
        'removed_resources': len(removed_resources),
        'added_prompts': len(added_prompts),
        'removed_prompts': len(removed_prompts),
    }

    result = {
        'ok': True,
        'passed': passed,
        'exit_code': 0 if passed else 1,
        'violations': violations,
        'notices': notices,
        'summary': summary,
        'diff': diff,
    }
    result['pretty'] = render_result('MCP Contract CI Gate', {
        'passed': passed,
        'exit_code': result['exit_code'],
        'violation_count': len(violations),
        'notice_count': len(notices),
        'baseline_hash': diff.get('baseline_hash', ''),
        'current_hash': diff.get('current_hash', ''),
    })
    return result


def mcp_policy_pack(
    profile: str = 'balanced',
    baseline_path: Optional[str] = None,
    provider: Optional[str] = None,
) -> Dict[str, Any]:
    """Tool: Build a governance policy pack for MCP build and release workflows."""
    profile_name = profile.strip().lower()
    policy_profiles = {
        'strict': {
            'allow_additive_contract': False,
            'provider_drift_fails': True,
            'controls': [
                {
                    'id': 'auth_visibility',
                    'level': 'required',
                    'expectation': 'Require auth for privileged tools and apply visibility boundaries by audience.',
                },
                {
                    'id': 'contract_gate',
                    'level': 'required',
                    'expectation': 'Fail CI on any contract drift, including additive tools/resources/prompts.',
                },
                {
                    'id': 'provider_drift',
                    'level': 'required',
                    'expectation': 'Fail if provider config is missing or drifted for target clients.',
                },
                {
                    'id': 'duplicate_reasoning',
                    'level': 'required',
                    'expectation': 'Require reasoning before dedupe: ABC implementations and backend-specific variants may be valid.',
                },
                {
                    'id': 'dead_code_weighting',
                    'level': 'required',
                    'expectation': 'Treat dead-code as weighted signal: lower-level dead exports are higher risk; tutorial usage lowers concern.',
                },
                {
                    'id': 'docs_freshness',
                    'level': 'required',
                    'expectation': 'Keep root or agent-scoped living docs current (ROADMAP, TODO, plan files).',
                },
                {
                    'id': 'tutorial_hygiene',
                    'level': 'required',
                    'expectation': 'tutorials/ is for runnable notebooks only, not cache/artifacts/outputs.',
                },
            ],
        },
        'balanced': {
            'allow_additive_contract': True,
            'provider_drift_fails': True,
            'controls': [
                {
                    'id': 'auth_visibility',
                    'level': 'required',
                    'expectation': 'Use auth/visibility boundaries for write or privileged tools.',
                },
                {
                    'id': 'contract_gate',
                    'level': 'required',
                    'expectation': 'Fail CI on removals or schema changes; allow additive surface area.',
                },
                {
                    'id': 'provider_drift',
                    'level': 'required',
                    'expectation': 'Fail if provider config is missing or drifted.',
                },
                {
                    'id': 'duplicate_reasoning',
                    'level': 'recommended',
                    'expectation': 'Use reasoning before merging similar functions; keep valid backend variants.',
                },
                {
                    'id': 'dead_code_weighting',
                    'level': 'recommended',
                    'expectation': 'Dead exports may be public API used in tutorials; prioritize by module hierarchy.',
                },
                {
                    'id': 'docs_freshness',
                    'level': 'recommended',
                    'expectation': 'Keep roadmap/changelog/plan docs in sync with behavior changes.',
                },
                {
                    'id': 'tutorial_hygiene',
                    'level': 'required',
                    'expectation': 'Store outputs at repo root or ~/Downloads/<repo>, not tutorials/.',
                },
            ],
        },
        'advisory': {
            'allow_additive_contract': True,
            'provider_drift_fails': False,
            'controls': [
                {
                    'id': 'auth_visibility',
                    'level': 'recommended',
                    'expectation': 'Prefer auth and visibility filters for non-readonly tools.',
                },
                {
                    'id': 'contract_gate',
                    'level': 'recommended',
                    'expectation': 'Run contract gate in CI, but report before enforcing failures.',
                },
                {
                    'id': 'provider_drift',
                    'level': 'recommended',
                    'expectation': 'Track provider drift regularly and reconcile before release.',
                },
                {
                    'id': 'duplicate_reasoning',
                    'level': 'recommended',
                    'expectation': 'Review duplicate reports with semantic intent before deleting code.',
                },
                {
                    'id': 'dead_code_weighting',
                    'level': 'recommended',
                    'expectation': 'Consider tutorial usage before marking exports as dead code.',
                },
                {
                    'id': 'docs_freshness',
                    'level': 'recommended',
                    'expectation': 'Keep living docs current when implementation or workflows change.',
                },
                {
                    'id': 'tutorial_hygiene',
                    'level': 'recommended',
                    'expectation': 'Avoid storing generated artifacts inside tutorials/.',
                },
            ],
        },
    }

    if profile_name not in policy_profiles:
        valid = ', '.join(sorted(policy_profiles.keys()))
        return {'ok': False, 'error': f'Unknown profile: {profile}. Valid: {valid}'}

    selected = policy_profiles[profile_name]
    checks: Dict[str, Any] = {}
    violations: List[str] = []
    warnings: List[str] = []

    if baseline_path:
        gate = mcp_contract_ci_gate(
            baseline_path=baseline_path,
            allow_additive_tools=selected['allow_additive_contract'],
            allow_additive_resources=selected['allow_additive_contract'],
            allow_additive_prompts=selected['allow_additive_contract'],
        )
        checks['contract_gate'] = {
            'ok': gate.get('ok', False),
            'passed': gate.get('passed', False),
            'exit_code': gate.get('exit_code', 1),
            'violation_count': len(gate.get('violations', [])) if gate.get('ok') else None,
        }
        if gate.get('ok') and not gate.get('passed', False):
            violations.extend(gate.get('violations', []))
        elif not gate.get('ok'):
            violations.append(f"Contract gate check failed to run: {gate.get('error', 'unknown error')}")
    else:
        checks['contract_gate'] = {
            'ok': True,
            'passed': None,
            'exit_code': None,
            'violation_count': None,
            'note': 'Skipped (no baseline_path provided).',
        }

    drift = mcp_provider_drift_report(provider=provider)
    if drift.get('ok'):
        drifted_count = drift.get('drifted_count', 0)
        checks['provider_drift'] = {
            'ok': True,
            'drifted_count': drifted_count,
            'providers_checked': len(drift.get('providers', [])),
        }
        if drifted_count > 0:
            if selected['provider_drift_fails']:
                violations.append(f"Provider drift detected for {drifted_count} provider(s).")
            else:
                warnings.append(f"Provider drift detected for {drifted_count} provider(s).")
    else:
        checks['provider_drift'] = {'ok': False, 'error': drift.get('error', 'unknown error')}
        if selected['provider_drift_fails']:
            violations.append(f"Provider drift check failed: {drift.get('error', 'unknown error')}")
        else:
            warnings.append(f"Provider drift check failed: {drift.get('error', 'unknown error')}")

    passed = len(violations) == 0
    result = {
        'ok': True,
        'profile': profile_name,
        'controls': selected['controls'],
        'checks': checks,
        'passed': passed,
        'violations': violations,
        'warnings': warnings,
        'next_steps': [
            'Run mcp_compatibility_matrix to validate provider readiness.',
            'Gate releases with mcp_contract_ci_gate and a committed baseline contract.',
            'Treat duplicate/dead-code reports as review signals, not automatic deletions.',
        ],
    }
    result['pretty'] = render_result('MCP Policy Pack', {
        'profile': profile_name,
        'passed': passed,
        'control_count': len(selected['controls']),
        'violation_count': len(violations),
        'warning_count': len(warnings),
    })
    return result


## MCP Registration

In [ ]:
#| export
# Tool annotation definitions for project tools
TOOL_ANNOTATIONS = {
    'set_project': ToolAnnotations(
        title="Set Project",
        readOnlyHint=False,
        idempotentHint=True,
        openWorldHint=False
    ),
    'current_project': ToolAnnotations(
        title="Current Project",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'console_scripts_status': ToolAnnotations(
        title="Console Scripts Status",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'find_projects': ToolAnnotations(
        title="Find Projects",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=True
    ),
    'bookmark_project': ToolAnnotations(
        title="Bookmark Project",
        readOnlyHint=False,
        idempotentHint=True,
        openWorldHint=False
    ),
    'list_bookmarks': ToolAnnotations(
        title="List Bookmarks",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'remove_bookmark': ToolAnnotations(
        title="Remove Bookmark",
        readOnlyHint=False,
        idempotentHint=True,
        openWorldHint=False
    ),
    'config_status': ToolAnnotations(
        title="Config Status",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'prompt_templates_status': ToolAnnotations(
        title="Prompt Templates Status",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'health_check': ToolAnnotations(
        title="Health Check",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'server_metrics': ToolAnnotations(
        title="Server Metrics",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'analyze_remote': ToolAnnotations(
        title="Analyze Remote",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=True
    ),
    'mcp_scaffold_guide': ToolAnnotations(
        title="MCP Scaffold Guide",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'scaffold_mcp_notebooks': ToolAnnotations(
        title="Scaffold MCP Notebooks",
        readOnlyHint=False,
        idempotentHint=False,
        openWorldHint=False
    ),
    'mcp_contract_snapshot': ToolAnnotations(
        title="MCP Contract Snapshot",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'mcp_contract_diff': ToolAnnotations(
        title="MCP Contract Diff",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'mcp_provider_drift_report': ToolAnnotations(
        title="MCP Provider Drift",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'mcp_composition_workbench': ToolAnnotations(
        title="MCP Composition Workbench",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'mcp_compatibility_matrix': ToolAnnotations(
        title="MCP Compatibility Matrix",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'mcp_contract_ci_gate': ToolAnnotations(
        title="MCP Contract CI Gate",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'mcp_policy_pack': ToolAnnotations(
        title="MCP Policy Pack",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
}

def add_project_tools(mcp: FastMCP) -> None:
    """Attach project management tools to the MCP server with annotations.

    Args:
        mcp: FastMCP server instance.
    """
    tools = [
        ('set_project', set_project),
        ('current_project', current_project),
        ('console_scripts_status', console_scripts_status),
        ('find_projects', find_projects),
        ('bookmark_project', bookmark_project),
        ('list_bookmarks', list_bookmarks),
        ('remove_bookmark', remove_bookmark),
        ('config_status', config_status),
        ('prompt_templates_status', prompt_templates_status),
        ('health_check', health_check),
        ('server_metrics', server_metrics),
        ('analyze_remote', analyze_remote),
        ('mcp_scaffold_guide', mcp_scaffold_guide),
        ('scaffold_mcp_notebooks', scaffold_mcp_notebooks),
        ('mcp_contract_snapshot', mcp_contract_snapshot),
        ('mcp_contract_diff', mcp_contract_diff),
        ('mcp_provider_drift_report', mcp_provider_drift_report),
        ('mcp_composition_workbench', mcp_composition_workbench),
        ('mcp_compatibility_matrix', mcp_compatibility_matrix),
        ('mcp_contract_ci_gate', mcp_contract_ci_gate),
        ('mcp_policy_pack', mcp_policy_pack),
    ]

    for name, func in tools:
        annotations = TOOL_ANNOTATIONS.get(name)
        mcp.tool(name=name, annotations=annotations)(func)


## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
#| export
import tempfile
import shutil

def analyze_remote(
    url: str,
    branch: str = "main",
    shallow: bool = True,
    timeout: Optional[int] = None
) -> Dict[str, Any]:
    """Tool: Analyze a remote nbdev project without permanent cloning.
    
    Clones a GitHub repository temporarily, analyzes its structure,
    and returns information about the project. Useful for comparing
    with local projects or understanding external nbdev projects.
    
    Args:
        url: GitHub URL (https://github.com/user/repo) or shorthand (user/repo).
        branch: Branch to analyze (default: main).
        shallow: If True, do a shallow clone for faster analysis.
        timeout: Timeout in seconds for cloning (default from config/environment).
    
    Returns:
        Result dict with project info, structure, and analysis.
    """
    import subprocess
    from nbdev_mcp.utils.config import get_config
    
    cfg = get_config()
    if timeout is None:
        timeout = cfg.timeout_analyze_remote
    
    # Normalize URL
    if not url.startswith('http'):
        url = f'https://github.com/{url}'
    
    # Extract repo name
    repo_name = url.rstrip('/').split('/')[-1]
    if repo_name.endswith('.git'):
        repo_name = repo_name[:-4]
    
    # Create temp directory
    temp_dir = tempfile.mkdtemp(prefix='nbdev_mcp_')
    clone_path = Path(temp_dir) / repo_name
    
    try:
        # Clone the repo
        cmd = ['git', 'clone']
        if shallow:
            cmd.extend(['--depth', '1'])
        cmd.extend(['-b', branch, url, str(clone_path)])
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        if result.returncode != 0:
            return {
                'ok': False,
                'error': f'Clone failed: {result.stderr}',
                'url': url,
                'branch': branch
            }

        # Check if it's an nbdev project
        if not is_nbdev_project(clone_path):
            return {
                'ok': False,
                'error': 'Not an nbdev project (expected nbs/ plus settings.ini/settings.toml/pyproject.toml [tool.nbdev])',
                'url': url
            }

        settings = settings_dict(clone_path)
        settings_file = nbdev_settings_path(clone_path)
        generation = nbdev_generation(clone_path)

        # Find notebooks
        nbs_path = settings.get('nbs_path', 'nbs')
        nbs_dir = clone_path / nbs_path

        notebooks = []
        if nbs_dir.exists():
            notebooks = sorted([
                str(nb.relative_to(clone_path))
                for nb in nbs_dir.rglob('*.ipynb')
                if not nb.name.startswith('.')
            ])

        # Find lib path
        lib_path = settings.get('lib_path', settings.get('lib_name', ''))
        lib_dir = clone_path / lib_path if lib_path else clone_path

        modules = []
        if lib_dir.exists():
            modules = sorted([
                str(m.relative_to(clone_path))
                for m in lib_dir.rglob('*.py')
                if not m.name.startswith('.')
            ])

        requirements_raw = settings.get('requirements', '')

        # Build analysis
        analysis = {
            'ok': True,
            'url': url,
            'branch': branch,
            'name': settings.get('lib_name', repo_name),
            'version': settings.get('version', 'unknown'),
            'description': settings.get('description', ''),
            'author': settings.get('author', ''),
            'requirements': requirements_raw.split() if requirements_raw else [],
            'nbdev_generation': generation,
            'nbdev_settings_file': settings_file.name if settings_file is not None else None,
            'notebooks': {
                'count': len(notebooks),
                'paths': notebooks[:50],  # Limit to 50
            },
            'modules': {
                'count': len(modules),
                'paths': modules[:50],
            },
            'settings': {k: v for k, v in settings.items() if k not in ['requirements', 'dev_requirements']},
        }
        
        from nbdev_mcp.utils.rich import render_result
        pretty = render_result(f'Remote Analysis: {repo_name}', {
            'name': analysis['name'],
            'version': analysis['version'],
            'nbdev_generation': analysis['nbdev_generation'],
            'settings_file': analysis['nbdev_settings_file'] or 'N/A',
            'notebooks': analysis['notebooks']['count'],
            'modules': analysis['modules']['count'],
            'description': analysis['description'][:100] if analysis['description'] else 'N/A'
        })
        
        return {**analysis, 'pretty': pretty}
        
    except subprocess.TimeoutExpired:
        return {'ok': False, 'error': 'Clone timed out', 'url': url}
    except Exception as e:
        return {'ok': False, 'error': str(e), 'url': url}
    finally:
        # Clean up temp directory
        try:
            shutil.rmtree(temp_dir)
        except Exception:
            pass


In [ ]:
#| export
import time

# Module-level tracking for server metrics
_SERVER_START_TIME: Optional[float] = None;
_REQUEST_COUNT: int = 0;
_TOTAL_LATENCY_MS: float = 0.0;

def _init_metrics():
    """Initialize server metrics on first call."""
    global _SERVER_START_TIME
    if _SERVER_START_TIME is None:
        _SERVER_START_TIME = time.time()

def _record_request(latency_ms: float):
    """Record a request's latency."""
    global _REQUEST_COUNT, _TOTAL_LATENCY_MS
    _REQUEST_COUNT += 1
    _TOTAL_LATENCY_MS += latency_ms

def server_metrics() -> Dict[str, Any]:
    """Tool: Return server performance and health metrics.
    
    Returns comprehensive server status including:
    - Uptime
    - Memory usage
    - Request statistics
    - System information
    
    Returns
    -------
    Dict[str, Any]
        Result with uptime, memory, request stats, and system info.
    """
    import sys
    import platform
    import os
    
    _init_metrics()
    
    # Calculate uptime
    uptime_seconds = time.time() - _SERVER_START_TIME if _SERVER_START_TIME else 0
    uptime_hours = uptime_seconds / 3600
    
    # Memory usage
    try:
        import psutil
        process = psutil.Process(os.getpid())
        memory_mb = process.memory_info().rss / (1024 * 1024)
        memory_percent = process.memory_percent()
        cpu_percent = process.cpu_percent()
    except ImportError:
        # psutil not available
        memory_mb = None
        memory_percent = None
        cpu_percent = None
    
    # Request stats
    avg_latency = _TOTAL_LATENCY_MS / _REQUEST_COUNT if _REQUEST_COUNT > 0 else 0
    
    # Build metrics
    metrics = {
        'ok': True,
        'uptime': {
            'seconds': round(uptime_seconds, 1),
            'hours': round(uptime_hours, 2),
            'started_at': time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(_SERVER_START_TIME)) if _SERVER_START_TIME else None
        },
        'requests': {
            'count': _REQUEST_COUNT,
            'avg_latency_ms': round(avg_latency, 2),
            'total_latency_ms': round(_TOTAL_LATENCY_MS, 2)
        },
        'memory': {
            'rss_mb': round(memory_mb, 1) if memory_mb else 'N/A (install psutil)',
            'percent': round(memory_percent, 1) if memory_percent else 'N/A',
        },
        'cpu': {
            'percent': round(cpu_percent, 1) if cpu_percent else 'N/A (install psutil)',
        },
        'system': {
            'python': sys.version.split()[0],
            'platform': platform.system(),
            'arch': platform.machine(),
            'pid': os.getpid()
        }
    }
    
    from nbdev_mcp.utils.rich import render_table, render_panel
    
    rows = [
        ['Uptime', f"{metrics['uptime']['hours']} hours"],
        ['Requests', str(metrics['requests']['count'])],
        ['Avg Latency', f"{metrics['requests']['avg_latency_ms']} ms"],
        ['Memory', f"{metrics['memory']['rss_mb']} MB" if memory_mb else 'N/A'],
        ['CPU', f"{metrics['cpu']['percent']}%" if cpu_percent else 'N/A'],
        ['Python', metrics['system']['python']],
        ['Platform', f"{metrics['system']['platform']} {metrics['system']['arch']}"],
    ]
    pretty = render_table('Server Metrics', ['Metric', 'Value'], rows)
    
    return {**metrics, 'pretty': pretty}